In [3]:
# Import Packages
import os
import re
import sys
import time
import copy
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import r2_score, mean_squared_error

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import cross_val_score, GroupKFold

from sklearn.ensemble import RandomForestRegressor

from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet

from sklearn.feature_selection import VarianceThreshold

from sklearn.feature_selection import f_classif

# GENERAL UTILITIES

In [ ]:
# IDENTIFY LOW AND HIGH AGB PLOTS
# Classifies all 59 plots into three groups:
#  Group-1: Near-zero variance plots whose R² scores are mathematically
#           unreliable as holdout sets
#  Group-2: High-AGB plots that are statistical outliers at either
#           the dataset level or within their own site,
#  Group-3: Everything else.
#
#Returns both plot-level and site-level categories for use in grouped CV evaluation.
def get_low_and_high_agb_plots(y, plot_groups):
    near_zero_plots, high_agb_plots = get_plot_categories(y, plot_groups)
    
    within_site_high_agb = get_within_site_high_agb_plots(y, plot_groups, near_zero_plots)
    high_agb_plots = list(set(high_agb_plots + within_site_high_agb))
    
    # Derive site-level categories from plot-level results
    near_zero_sites = list(set(
        plot.rsplit('_', maxsplit=1)[0]
        for plot in near_zero_plots
    ))
    
    # Exclude near-zero sites from high-AGB sites
    # Big Creek_3 is high-AGB at plot level but Big Creek as a site
    # is near-zero — site level takes precedence here
    high_agb_sites = list(set(
        plot.rsplit('_', maxsplit=1)[0]
        for plot in high_agb_plots
        if plot.rsplit('_', maxsplit=1)[0] not in near_zero_sites
    ))
    
    print("\nNear-zero sites:", sorted(near_zero_sites))
    print("High-AGB sites :", sorted(high_agb_sites))
    return near_zero_sites, high_agb_sites, near_zero_plots, high_agb_plots

In [ ]:
# IDENTIFY LOW AND HIGH AGB SITES (SITE-LEVEL GROUPED CV)
# - Same classification logic as get_low_and_high_agb_plots but operates at site
#   level rather than plot level.
# - Used when grouped CV holds out entire sites.
# - Near-zero sites are detected via the largest relative gap in log variance
#   across sites.
# - High-AGB sites are detected via IQR on site-level max AGB.
def get_site_categories(y, groups):
    site_groups = groups.map(lambda x: x.rsplit('_', maxsplit=1)[0])
    site_stats  = y.groupby(site_groups).agg(
        max_agb = 'max',
        var_agb = 'var',
        n_rows  = 'count'
    )

    # ── High-AGB sites: IQR on site-level max AGB ─────────────────
    max_agb_values = site_stats['max_agb']
    Q1             = max_agb_values.quantile(0.25)
    Q3             = max_agb_values.quantile(0.75)
    IQR            = Q3 - Q1
    high_threshold = Q3 + 1.5 * IQR
    high_agb_sites = site_stats[max_agb_values > high_threshold].index.tolist()

    # ── Near-zero sites: largest relative gap in log variance ──────
    log_var         = np.log1p(y).groupby(site_groups).var()
    log_var_sorted  = log_var.sort_values()
    relative_gaps   = log_var_sorted.diff() / log_var_sorted.shift(1)
    largest_gap_idx = relative_gaps.idxmax()
    gap_position    = log_var_sorted.index.get_loc(largest_gap_idx)
    near_zero_threshold = log_var_sorted.iloc[gap_position - 1]
    near_zero_sites = log_var[log_var <= near_zero_threshold].index.tolist()

    print(f"High-AGB threshold  (Q3 + 1.5*IQR)       : {high_threshold:.2f} kg")
    print(f"Near-zero threshold (largest relative gap): {near_zero_threshold:.6f}")

    print(f"\nNear-zero variance sites:")
    for site in near_zero_sites:
        print(f"  {site:20s} : log var = {log_var[site]:.6f}")

    print(f"\nHigh-AGB sites:")
    for site in high_agb_sites:
        print(f"  {site:20s} : max AGB = {site_stats.loc[site, 'max_agb']:.1f} kg")

    return near_zero_sites, high_agb_sites

In [6]:
# IDENTIFY LOW AND HIGH AGB PLOTS (DATASET-LEVEL)
# Classifies plots using two data-driven thresholds.
#  - High-AGB plots are those those max AGB exceeds Q3 + 1.5*IQR
#    across all 59 plots — capturing Channel Caye and New River plots
#    as global outliers.
#  - Near-zero plots are those below the largest relative jump in 
#    log variance across plots — capturing Frenchman Caye, Shipstern, 
#    and Big Creek plots where R² is mathematically unreliable.
def get_plot_categories(y, groups):
    plot_stats = y.groupby(groups).agg(max_agb = 'max',
                                       var_agb = 'var',
                                       n_rows  = 'count')

    # High-AGB plots: IQR on plot-level max AGB
    max_agb_values = plot_stats['max_agb']
    Q1             = max_agb_values.quantile(0.25)
    Q3             = max_agb_values.quantile(0.75)
    IQR            = Q3 - Q1
    high_threshold = Q3 + 1.5 * IQR
    high_agb_plots = plot_stats[
        max_agb_values > high_threshold
    ].index.tolist()

    # Near-zero plots: largest relative gap in log variance
    log_var         = np.log1p(y).groupby(groups).var()
    log_var_sorted  = log_var.sort_values()
    relative_gaps   = log_var_sorted.diff() / log_var_sorted.shift(1)
    largest_gap_idx = relative_gaps.idxmax()
    gap_position    = log_var_sorted.index.get_loc(largest_gap_idx)
    near_zero_threshold = log_var_sorted.iloc[gap_position - 1]
    near_zero_plots = log_var[log_var < near_zero_threshold].index.tolist()

    print(f"High-AGB threshold  : {high_threshold:.2f} kg")
    print(f"Near-zero threshold : {near_zero_threshold:.6f}")

    print(f"\nNear-zero variance plots:")
    for plot in near_zero_plots:
        print(f"  {plot:25s} : log var = {log_var[plot]:.6f}")

    print(f"\nHigh-AGB plots:")
    for plot in high_agb_plots:
        print(f"  {plot:25s} : max AGB = {plot_stats.loc[plot, 'max_agb']:.1f} kg")

    return near_zero_plots, high_agb_plots

In [ ]:
# IDENTIFY HIGH AGB PLOTS WITHIN EACH SITE
#  - Finds plots that are AGB outliers relative to other plots in the same site.
#  - These are missed by the dataset-level IQR because their AGB is not extreme
#    globally — only within their site.
#  - Example: Big Creek_3 (max 12.6 kg) is not a global outlier but is well above
#    the other Big Creek plots (max 6-8 kg).
#  - Near-zero plots are excluded before the within-site IQR is computed to prevent
#    sites like Big Creek from being skipped entirely due to their near-zero plots.
def get_within_site_high_agb_plots(y, plot_groups,
                                   near_zero_plots,
                                   iqr_multiplier=1.5):
    site_groups    = plot_groups.map(lambda x: x.rsplit('_', maxsplit=1)[0])
    near_zero_set  = set(near_zero_plots)  # plot-level, not site-level

    print("\nwithin-site high-AGB plots")
    within_site_outliers = []

    for site in site_groups.unique():
        site_mask = site_groups == site
        plot_max  = y[site_mask].groupby(plot_groups[site_mask]).max()

        # Exclude near-zero plots from within-site comparison
        plot_max  = plot_max[~plot_max.index.isin(near_zero_set)]

        if len(plot_max) < 3:
            continue

        Q1             = plot_max.quantile(0.25)
        Q3             = plot_max.quantile(0.75)
        IQR            = Q3 - Q1
        high_threshold = Q3 + iqr_multiplier * IQR

        outlier_plots  = plot_max[plot_max > high_threshold].index.tolist()
        if outlier_plots:
            print(f"{site:20s} : within-site high-AGB plots = {outlier_plots}")
            within_site_outliers.extend(outlier_plots)

    return within_site_outliers

In [ ]:
def split_data_by_groups(X_var, y_var, groups):

    if len(groups.unique()) < 2:
        return split_data_ver2(X_var, y_var)
    
    X_local = X_var.copy(deep=True)
    y_local = y_var.copy(deep=True)

    # Let us say there are 5 unique plots and 15 trees in total, aka, 3 trees per plot.
    #  Plot A — rows 0, 1, 2
    #  Plot B — rows 3, 4, 5
    #  Plot C — rows 6, 7, 8
    #  Plot D — rows 9, 10, 11
    #  Plot E — rows 12, 13, 14
    #
    # GroupShuffleSplit
    # -----------------
    # This is a cross-validation iterator that generates train/test indices
    # to split data into subsets, ensuring specific groups do not overlap
    # between sets.
    # 
    # n_splits=1 => One fixed split. 80-20 split in this case.
    # OUTPUT of GroupShuffleSplit(n_splits=1..)
    #   train_idx = [0, 1, 2, 3, 4, 5, 9, 10, 11, 12, 13, 14] # plots A,B,D,E
    #   test_idx  = [6, 7, 8]                                 # plot C only
    #
    # next(gss.split())
    # -----------------
    # gss.split() is a generator that would produce n_splits pairs.
    # Since n_splits=1 there is only one pair to produce.
    # The next() call retrieves that one pair and returns it.
    # OUTPUT of next(gss.split())
    #    train_idx = [0,1,2,3,4,5,9,10,11,12,13,14]
    #    test_idx  = [6,7,8]
    #
    # Interpolation (The "Cheat" Split)
    # ---------------------------------
    #  If the training set has 8 trees from Plot_A and the test set has the
    #  remaining 2 trees from Plot_A, the  model is interpolating. It already
    #  knows the specific "vibe" of Plot_A.
    #
    #  Result:
    #   - High R2 that is false.
    #   - The model is just identifying trees it already "knows" by proxy.
    #
    # Extrapolation (The "True" Split)
    # ---------------------------------
    #  If you put every single tree from Plot_A into the test set and train the model
    #  only on trees from Plot_B, C, and D, the model is extrapolating.
    #
    #  The "New Site": In this context, Plot_A is the "new site."
    #  It is a geographic location the model has never seen.
    #
    #  Result:
    #    - A lower, but realistic R2.
    #    - This tells you how well your model will perform if you take it to a
    #      completely different part of Panama or Brazil.
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

    train_idx, test_idx = next(gss.split(X_local, y_local, groups=groups))
    
    X_train = X_local.iloc[train_idx]
    X_test  = X_local.iloc[test_idx]
    y_train = y_local.iloc[train_idx]
    y_test  = y_local.iloc[test_idx]

    train_plots = set(groups.iloc[train_idx]) #plot_id
    test_plots  = set(groups.iloc[test_idx])
    
    overlap     = train_plots & test_plots
    #print(f"Train plots : {len(train_plots)}")
    #print(f"Test plots  : {len(test_plots)}")
    #print(f"Overlapping plots: {len(overlap)}")  # must be 0
    assert not len(overlap)
    
    return X_train, X_test, y_train, y_test

In [ ]:
def split_data_ver2(X_var, y_var):
    # The train_test_split() splits randomly by rows — it
    # does not know about any groups (e.g., plot ids, etc).
    # 
    # When the data is split with this method, trees from the same plot
    # can end up in both train and test. The model trains on 58 trees
    # from plot X and is tested on the remaining 7 trees from the same plot.
    # This is pseudo-replication.
    #
    # pseudo-replication
    # ------------------
    # Pseudo-replication occurs when individual data points are treated as
    # independent observations in a statistical model, even though they are
    # actually clustered or correlated due to a shared environment.

    # When is pseudo-replication NOT useful?
    #  - Pseudo-replication is a problem specifically when your data has a
    #    hierarchical structure, e.g., trees nested within plots,
    #    students nested within schools, etc.
    #  - In those cases observations within the same group are correlated and
    #   random splitting leaks information.
    # E.g.
    #  - When trees from the same plot appear in both train and test,
    #    the model is evaluated on data it has effectively already seen,
    #    i.e., same EMIT pixel, same site conditions, same species mix.
    #  - The R² will be inflated.
    #  - You report a number that does not reflect real world performance on new sites.

    # When is pseudo-replication useful?
    #  - Pseudo-replication is acceptable when your dataset has no meaningful
    #    grouping structure, aka, the observations are independent of each other.

    y_quantiles = pd.qcut(y_var, q=4, labels=False)
    X_train, X_test, y_train, y_test = train_test_split(X_var,
                                                        y_var,
                                                        test_size=0.2,
                                                        random_state=42,
                                                        stratify=y_quantiles)

    return X_train, X_test, y_train, y_test

# EVALUATION FUNCTIONS

In [ ]:
def get_site_from_plot(plot_name):
    return plot_name.rsplit('_', maxsplit=1)[0]

In [5]:
def evaluate_experiment(label, results,
                        fold_sites=None,
                        fold_plots=None,
                        near_zero_sites=None,
                        high_agb_sites=None,
                        near_zero_plots=None,
                        high_agb_plots=None):

    near_zero_sites = near_zero_sites or []
    high_agb_sites  = high_agb_sites  or []
    near_zero_plots = near_zero_plots or []
    high_agb_plots  = high_agb_plots  or []

    test_r2   = results.get('test_r2')
    test_rmse = results.get('test_rmse')
    cv_mean   = results.get('cv_r2_mean')
    cv_std    = results.get('cv_r2_std')
    cv_scores = results.get('cv_scores')

    group_cv_mean   = results.get('group_cv_r2_mean')
    group_cv_std    = results.get('group_cv_r2_std')
    group_cv_scores = results.get('group_cv_scores')

    is_grouped = group_cv_scores is not None and fold_sites is not None

    print("=" * 60)
    print(f"EXPERIMENT EVALUATION: {label}")
    print("=" * 60)

    print(f"\nTest set:")
    print(f"  R²   : {test_r2:.3f}")
    print(f"  RMSE : {test_rmse:.2f} kg")

    print(f"\nRegular CV:")
    print(f"  Mean   : {cv_mean:.3f}")
    print(f"  Std    : {cv_std:.3f}")
    print(f"  Scores : {np.round(cv_scores, 3)}")

    checks = []

    if test_r2 > 0:
        print(f"\n  ✅ Test R² is positive ({test_r2:.3f})")
        checks.append(True)
    else:
        print(f"\n  ❌ Test R² is negative ({test_r2:.3f})")
        checks.append(False)

    if cv_mean > 0:
        print(f"  ✅ CV mean is positive ({cv_mean:.3f})")
        checks.append(True)
    else:
        print(f"  ❌ CV mean is negative ({cv_mean:.3f})")
        checks.append(False)

    if is_grouped:
        scores = np.array(group_cv_scores)

        print(f"\nGrouped CV:")
        # ── Per-fold breakdown with contamination check ───────────
        for i, (site, score) in enumerate(zip(fold_sites, scores)):
            if score < 0:
                contaminated = fold_plots is not None and any(
                    p != site and p in high_agb_plots
                    for p in fold_plots[i]
                )
                if contaminated:
                    contam_plot = next(
                        p for p in fold_plots[i]
                        if p != site and p in high_agb_plots
                    )
                    reason = f"⚠️  fold contamination — {contam_plot} in same fold"
                elif site in high_agb_plots:             # ← plot-level check first
                    reason = "❌ genuine generalization failure"
                elif site in near_zero_plots:            # ← plot-level near-zero
                    reason = "⚠️  near-zero variance artifact"
                elif get_site_from_plot(site) in near_zero_sites:  # ← site-level fallback
                    reason = "⚠️  near-zero variance artifact"
                else:
                    reason = "❌ negative — cause unknown"
            else:
                reason = "✅"
            print(f"  {site:20s} : {score:>7.3f}  {reason}")

        print(f"\n  Mean : {group_cv_mean:.3f}")
        print(f"  Std  : {group_cv_std:.3f}")

        # Check 3 — grouped CV mean positive
        if group_cv_mean > 0:
            print(f"\n  ✅ Grouped CV mean is positive ({group_cv_mean:.3f})")
            checks.append(True)
        else:
            print(f"\n  ❌ Grouped CV mean is negative ({group_cv_mean:.3f})")
            checks.append(False)

        # Check 4 — unexplained negatives ─────────────────────────
        unexplained = []
        for i, (site, score) in enumerate(zip(fold_sites, scores)):
            if score >= 0:
                continue
            site_name    = get_site_from_plot(site)
            contaminated = fold_plots is not None and any(
                p != site and p in high_agb_plots
                for p in fold_plots[i]
            )
            if (not contaminated
                    and site not in near_zero_plots
                    and site_name not in near_zero_sites
                    and site not in high_agb_plots):
                unexplained.append(site)

        if not unexplained:
            print(f"  ✅ All negative folds explained")
            checks.append(True)
        else:
            print(f"  ❌ Unexplained negative folds: {unexplained}")
            checks.append(False)

        # Check 5 — high AGB sites positive
        high_agb_failures = [
            site for site, score in zip(fold_sites, scores)
            if (get_site_from_plot(site) in high_agb_sites
                or site in high_agb_plots)
            and score < 0
            and site not in near_zero_plots
            and get_site_from_plot(site) not in near_zero_sites
        ]
        if not high_agb_failures:
            print(f"  ✅ High-AGB sites generalize correctly")
            checks.append(True)
        else:
            print(f"  ❌ High-AGB sites failed: {high_agb_failures}")
            checks.append(False)

    print(f"\n{'─' * 60}")
    grouped_cv_clean = is_grouped and all(checks[2:])  # all grouped checks pass
    
    if all(checks):
        print("VERDICT: ✅ ACCEPTABLE")
    elif grouped_cv_clean and not checks[1]:
        # Regular CV fails but grouped CV is fully clean
        print("VERDICT: ✅ ACCEPTABLE (grouped CV clean — regular CV unreliable)")
    elif checks[0] and checks[1]:
        print("VERDICT: ⚠️  MARGINAL — positive metrics but concerns")
    else:
        print("VERDICT: ❌ NOT ACCEPTABLE")

# VISUALIZATION

In [25]:
def plot_correlation_matrix(X, y, top_n=20, figsize=(12, 8)):
    df_corr = X.copy()
    df_corr['plant_AGB_kg'] = y.values

    target_corr = df_corr.corr()['plant_AGB_kg'].drop('plant_AGB_kg').abs().sort_values(ascending=False)
    
    # Compute correlation of all features against target only
    target_corr = df_corr.corr()['plant_AGB_kg'].drop('plant_AGB_kg')
    target_corr = target_corr.abs().sort_values(ascending=False).head(top_n)

    # Plot
    fig, ax = plt.subplots(figsize=figsize)
    target_corr.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'Top {top_n} Feature Correlations with plant_AGB_kg')
    ax.set_xlabel('Feature')
    ax.set_ylabel('Absolute Correlation')
    ax.axhline(y=0.1, color='red', linestyle='--', label='threshold=0.1')
    ax.legend()
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    return target_corr

In [2]:
def tabulate_results(results):
    rows = []
    has_group_cv = any(metrics.get('group_cv_r2_mean') is not None 
                       for metrics in results.values())
    
    for label, metrics in results.items():
        row = {
            'Experiment' : label,
            'Num_rows': round(metrics['num_rows'], 4),
            'Num_features': round(metrics['num_features'], 4),
            'Test R²'    : round(metrics['test_r2'], 4),
            'Test RMSE'  : round(metrics['test_rmse'], 2),
            'Train R²'   : round(metrics['train_r2'], 4),
            'Train RMSE' : round(metrics['train_rmse'], 2),
            'CV R² Mean' : round(metrics['cv_r2_mean'], 4),
            'CV R² Std'  : round(metrics['cv_r2_std'], 4),
        }
        if has_group_cv:
            row['Group CV R² Mean'] = round(metrics.get('group_cv_r2_mean', float('nan')), 4)
            row['Group CV R² Std']  = round(metrics.get('group_cv_r2_std', float('nan')), 4)
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    sort_col = 'Group CV R² Mean' if has_group_cv else 'CV R² Mean'
    df = df.sort_values(sort_col, ascending=False).reset_index(drop=True)
    
    highlight_max_cols = ['Test R²', sort_col]
    highlight_min_cols = ['Test RMSE']
    if has_group_cv:
        highlight_min_cols.append('Group CV R² Std')
    
    styled = (
        df.style
        .highlight_max(subset=highlight_max_cols, color='lightgreen')
        .highlight_min(subset=highlight_min_cols, color='lightgreen')
    )
    display(styled)

In [ ]:
def residual_analysis(y_pred, residuals):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].scatter(y_pred, residuals, alpha=0.4)
    axes[0].axhline(0, color='red', linestyle='--')
    axes[0].set_xlabel('Predicted AGB')
    axes[0].set_ylabel('Residuals')
    axes[0].set_title('Residuals vs Predicted')
    
    axes[1].hist(residuals, bins=30)
    axes[1].set_xlabel('Residual')
    axes[1].set_title('Residual Distribution')
    
    from scipy import stats
    stats.probplot(residuals, dist="norm", plot=axes[2])
    axes[2].set_title('QQ Plot')
    
    plt.tight_layout()
    plt.show()

In [ ]:
def show_importances(results, N=10):
    importances = results["importances"].sort_values(ascending=False)
    
    # Feature importances
    print(f"\nTop {N} feature importances:")
    for feat, imp in importances.head(N).items():
        bar = '█' * int(imp * 50)
        print(f"  {feat:45s} {imp:.4f}  {bar}")

# DATA CLEANING FUNCTIONS

In [17]:
def get_categorical_cols(X_data):
    return X_data.select_dtypes(include=['object', 'category']).columns.tolist()

def get_numerical_cols(X_data):
    return X_data.select_dtypes(include=['number']).columns.tolist()

In [22]:
def irrelevant_categorical_features(X, y, alpha=0.05, min_samples=2):
    """
    Calls categorical_matters for every categorical column.
    Drops columns that do not significantly affect the target.
    Returns filtered X and summary dataframe.
    """
    X_filtered       = X.copy()
    categorical_cols = get_categorical_cols(X)
    
    results      = []
    cols_to_drop = []

    for col in categorical_cols:
        print(f"\n--- {col} ---")
        relevance = categorical_relevance(X, y,
                                          col,
                                          alpha=alpha,
                                          min_samples=min_samples)

        results.append({'feature': col, 'relevant?': relevance})

        if not relevance:
            cols_to_drop.append(col)

    summary = pd.DataFrame(results)
    return cols_to_drop, summary

In [8]:
def remove_low_variance_cols(X_data,
                             exclude_cols,
                             threshold,
                             debug=False,
                             exclude_categorical=True):
    if exclude_categorical:
        categorical_cols = X_data.select_dtypes(include=['object', 'category']).columns.tolist()
        continuous_cols = [c for c in X_data.columns if c not in categorical_cols and c not in exclude_cols]
    else:
        continuous_cols = [c for c in X_data.columns if c not in exclude_cols]
    
    # Apply variance threshold
    X_continuous = X_data[continuous_cols]
    selector     = VarianceThreshold(threshold=threshold)
    selector.fit(X_continuous)
    
    low_variance_cols = X_continuous.columns[~selector.get_support()].tolist()
    print(f"Total low variance columns removed: {len(low_variance_cols)}")
    if debug:
        print(f"Low variance columns removed: {low_variance_cols}")
    
    kept_continuous = X_continuous.columns[selector.get_support()].tolist()
    
    # Reassemble — keep all encoded columns, keep only high-variance continuous columns
    features = None
    if exclude_categorical:
        features = kept_continuous + categorical_cols + exclude_cols
    else:
        features = kept_continuous + exclude_cols
    
    print(f"Features after variance filtering: {len(X_data.columns)}")
    return X_data[features]

In [24]:
def remove_uncorrelated_numerical_cols(X_data, y_data,
                                       threshold,
                                       exclude_cols,
                                       debug=False):
    categorical_cols  = X_data.select_dtypes(
                            include=['object', 'category']
                        ).columns.tolist()
    continuous_cols   = [c for c in X_data.columns
                         if c not in categorical_cols
                         and c not in exclude_cols]
    X_continuous      = X_data[continuous_cols]

    # Drop zero-variance columns before computing correlation
    # corrwith divides by std — zero std produces NaN and RuntimeWarning
    zero_var_cols     = X_continuous.columns[X_continuous.var() == 0].tolist()
    if zero_var_cols:
        print(f"Zero variance columns skipped: {zero_var_cols}")
    X_continuous      = X_continuous.drop(columns=zero_var_cols)

    correlations      = X_continuous.corrwith(y_data).abs()\
                                    .sort_values(ascending=False)
    strong_corr_cols  = correlations[correlations >= threshold].index.tolist()
    weak_corr_cols    = correlations[correlations <  threshold].index.tolist()

    if debug:
        print(f"Strong correlations kept : {strong_corr_cols}")
    print(f"Weak correlations removed: {weak_corr_cols}")

    X_data = X_data[strong_corr_cols + categorical_cols + exclude_cols]
    print(f"\nTotal features after correlation filtering: {len(X_data.columns)}")
    return X_data

In [21]:
from scipy.stats import f_oneway

def categorical_relevance(X, y, categorical_col, alpha=0.05, min_samples=2):
    # Get unique categories
    categories = X[categorical_col].unique()

    # Group target values by category.
    
    # Also, filter out groups with insufficient samples
    # Why?
    # ----
    #  ANOVA requires at least 2 samples per group to compute within-group variance.
    #  If any species category has only 1 tree — or zero trees in a particular subset
    #  of the data, the F-statistic cannot be computed and returns NaN.
    
    groups = [y[X[categorical_col] == cat].values for cat in categories
              if len(y[X[categorical_col] == cat]) >= min_samples]
    if len(groups) < 2:
        print(f"{categorical_col} : skipped — fewer than 2 valid groups after filtering")
        return False
    
    # One-way ANOVA
    f_stat, p_value = f_oneway(*groups)
    
    # What F-statistic Means
    # ----------------------
    # F = variance between species groups / variance within species groups
    # High F
    #  E.g.
    #   — AGB varies more between species than within species.
    #   - Species identity is a meaningful predictor of AGB.
    # Low F (close to 1)
    #  E.g.
    #   — AGB varies as much within species as between species.
    #   - Species identity adds little predictive value.

    # What does p-value Means?
    # -------------------
    #  It answers what is the probability of observing this F-statistic by chance
    #  if species had no effect on AGB?
    # p < 0.05   → species has a statistically significant effect on AGB
    # p > 0.05   → cannot conclude species affects AGB

    relevant = p_value < alpha
    
    print(f"Variable  : {categorical_col}")
    print(f"F-stat    : {f_stat:.4f}")
    print(f"p-value   : {p_value:.4f}")
    print(f"Relevant? : {relevant}")
    
    return relevant

def remove_uncorrelated_categorical_cols(X_data, y_data):
    cols_to_drop, summary = irrelevant_categorical_features(X_data, y_data)
    print(f"\n{summary}")
    print("\n")
    print(f"Strong correlations kept   : {len(X_data.columns) - len(cols_to_drop)}")
    print(f"Weak correlations removed: {len(cols_to_drop)}")

    return X_data.drop(columns=cols_to_drop)

In [ ]:
def handle_null_columns(X_data):
    null_rows = X_data[X_data.isnull().any(axis=1)]
    total_nulls = X_data.isnull().sum().sum()
    
    print(f"Total NULL count           : {total_nulls}")
    print(f"Rows with at least one NULL: {len(null_rows)}")
    print(f"Total rows                 : {len(X_data)}")
    print(f"Percentage                 : {len(null_rows)/len(X_data)*100:.1f}%")
    
    # NULL count per column for only the affected rows
    null_col_counts = null_rows.isnull().sum().sort_values(ascending=False)
    
    print("\nNULL count per column in affected rows:")
    print(null_col_counts[null_col_counts > 0])

    vapor_bands = null_rows.isnull().sum()
    vapor_bands = vapor_bands[vapor_bands > 0].index.tolist()
    
    print(f"Dropping {len(vapor_bands)} columns:")
    print(vapor_bands)
    
    X_data = X_data.drop(columns=vapor_bands)
    print(f"\nNULL count after dropping: {X_data.isnull().sum().sum()}")

    return X_data

# CHECK OVERFIT

In [3]:
def get_fold_sites(X, y_log, plot_groups, n_splits=10):
    n_splits   = min(n_splits, plot_groups.nunique())
    gkf        = GroupKFold(n_splits=n_splits)
    fold_sites = []
    fold_plots = []  # full composition per fold

    for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X, y_log, plot_groups)
    ):
        plots_in_fold = plot_groups.iloc[test_idx].unique().tolist()
        # Primary plot — first non-near-zero, non-trivial plot
        primary = plots_in_fold[0]
        fold_sites.append(primary)
        fold_plots.append(plots_in_fold)

    return fold_sites, fold_plots

In [9]:
def cross_validate(model_func, X_data, y_data, cv=10, scoring='r2', display=True):
    cv_scores = cross_val_score(model_func,
                                X_data, y_data,
                                cv=cv,
                                scoring=scoring)
    if display:
        print("\n Cross-validation ---")
        print(f"CV R² mean: {cv_scores.mean():.4f}")
        print(f"CV R² std : {cv_scores.std():.4f}")
        print(f"CV scores : {cv_scores.round(3)}")

    return cv_scores.mean(), cv_scores.std(), cv_scores

In [ ]:
def cross_validate_by_groups(model_func, X_data, y_data,
                             groups, n_splits=10, scoring='r2',
                             display=True):
    # Cap n_splits to the number of unique groups
    n_unique_groups = len(groups.unique())

    # Example:
    # n_unique_groups = 60
    #  GroupKFold(n_splits=60) => 1 group held out per fold  (leave-one-group-out)
    #  GroupKFold(n_splits=10) => 6 groups held out per fold
    #  GroupKFold(n_splits=6)  => 10 groups held out per fold
    #  GroupKFold(n_splits=2)  => 30 groups held out per fold
    n_splits = min(n_splits, n_unique_groups) # num folds <= group count

    # GroupKFold splits data into K folds ensuring all rows belonging
    # to the same group are entirely in one fold.
    # Example:
    #   Num plots = 5, n_splits = 5
    #   GroupKFold assigns each plot to exactly one fold:
    #    Fold 1 — test: Plot A    train: Plots B, C, D, E
    #    Fold 2 — test: Plot B    train: Plots A, C, D, E
    #    Fold 3 — test: Plot C    train: Plots A, B, D, E
    #    Fold 4 — test: Plot D    train: Plots A, B, C, E
    #    Fold 5 — test: Plot E    train: Plots A, B, C, D
    gkf       = GroupKFold(n_splits=n_splits)
    cv_scores = cross_val_score(model_func,
                                X_data, y_data,
                                cv=gkf.split(X_data, y_data, groups),
                                scoring=scoring)

    if display:
        print("\nGrouped Cross-validation ---")
        print(f"Grouped CV R² mean: {cv_scores.mean():.4f}")
        print(f"Grouped CV R² std : {cv_scores.std():.4f}")
        print(f"Grouped CV scores : {cv_scores.round(3)}")

    return cv_scores.mean(), cv_scores.std(), cv_scores

In [ ]:
def select_best_model(experiments):
    max_val = max(v["cv_r2_mean"] for v in experiments.values())
    
    rf_best_labels = [
        label
        for label, vals in experiments.items()
        if vals["cv_r2_mean"] == max_val
    ]
    
    print(rf_best_labels)
    return rf_best_labels[0]

# MODELING FUNCTIONS

## GENERAL MODELING FUNCTIONS

In [1]:
def post_processing(model,
                    X_train, X_test,
                    y_train, y_test, y_train_log,
                    cv, cv_func, groups,
                    label,
                    display=True):
    # Test set performance
    y_pred_log       = model.predict(X_test)
    y_train_pred_log = model.predict(X_train)
    
    # Inverse log
    y_pred       = np.expm1(y_pred_log)
    y_train_pred = np.expm1(y_train_pred_log)
    
    test_r2    = r2_score(y_test, y_pred)
    test_rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
    
    train_r2   = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    
    residuals  = y_test - y_pred

    if display:
        print(f"\n {label}")
        print(f"Test R²     : {test_r2:.4f}")
        print(f"Test RMSE   : {test_rmse:.2f} kg")
        print(f"Train R² (log scale): {r2_score(y_train_log, y_train_pred_log):.4f}")
        print(f"Train R² (orig scale): {train_r2:.4f}")
        print(f"Train RMSE  : {train_rmse:.2f} kg")
        print(f"Num Features: {X_train.shape[1]}")

    # Cross-validation to confirm that the R^2 above is not by coincidence.
    cv_r2_mean, cv_r2_std, cv_scores = cross_validate(cv_func,
                                                      X_train,
                                                      y_train_log,
                                                      cv=cv,
                                                      scoring='r2',
                                                      display=display)
    
    datum = {
            "num_rows": X_train.shape[0],
            "num_features":X_train.shape[1],
            "test_r2": test_r2,
             "test_rmse": test_rmse,
             "train_r2": train_r2,
             "train_rmse": train_rmse,
             "y_pred": y_pred,
             "residuals": residuals,
             "cv_r2_mean": cv_r2_mean,
             "cv_r2_std": cv_r2_std,
             "cv_scores": cv_scores,
             "model": model}

    if groups is not None and len(groups.unique()) > 1:
        group_cv_r2_mean, group_cv_r2_std, group_cv_scores = \
            cross_validate_by_groups(cv_func,
                                     X_train,
                                     y_train_log,
                                     groups,
                                     n_splits=10,
                                     scoring='r2',
                                     display=display)

        fold_sites, fold_plots = get_fold_sites(X_train, y_train_log, groups)
        
        group_info = {"group_cv_r2_mean": group_cv_r2_mean,
                      "group_cv_r2_std": group_cv_r2_std,
                      "group_cv_scores": group_cv_scores,
                      "fold_sites": fold_sites,
                      "fold_plots": fold_plots}

        datum.update(group_info)

    return datum

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## LINEAR REGRESSION FUNCTIONS

In [ ]:
def linear_reg_regular(model_type, X_var, y_var, cv, label,display=True):
    X_train, X_test, y_train, y_test = split_data_ver2(X_var, y_var)
    return linear_reg(model_type,
                      X_train, X_test,
                      y_train, y_test,
                      cv, None,
                      label,display=True)

In [ ]:
# Group-aware linear regression with a single train/test split.
#
# Splits data using GroupShuffleSplit (80/20) ensuring no group appears
# in both train and test sets. Trains on the train split and evaluates
# on the held-out group(s). Grouped CV is performed on the training set
# to assess within-train generalization across groups.      
def linear_reg_groups(model_type, X_var, y_var, cv, groups, label,display=True):
    X_train, X_test, y_train, y_test = split_data_by_groups(X_var, y_var, groups)
    
    # Split groups the same way as X and y
    groups_train = groups[y_train.index]
    groups_test  = groups[y_test.index]
    
    return linear_reg(model_type,
                      X_train, X_test,
                      y_train, y_test,
                      cv, groups_train,
                      label,display=True)

In [ ]:
#Leave-One-Group-Out (LOGO) linear regression evaluation.
#
# Trains and evaluates a linear model by iterating over all unique groups,
# holding out one complete group as the test set each time. Only folds with
# at least 2 test samples are evaluated. Final test_r2 and test_rmse are
# the mean across all valid folds.
#
# How is this different compared to linear_reg_groups?
#  linear_reg_groups gives you one honest split
#  linear_reg_logo gives you N honest splits
def linear_reg_logo(model_type, X_var, y_var, cv, groups, label):
    splits       = split_data_by_groups_logo(X_var, y_var, groups)
    fold_results = {}

    for X_train, X_test, y_train, y_test in splits:
        dataset_name = groups[y_test.index].iloc[0]
        print(f"  fold: {dataset_name}, y_test size: {len(y_test)}")  # ADD THIS
        if len(y_test) < 2:
            continue
        groups_train = groups[y_train.index]
        
        # Cap n_splits to number of unique training groups
        effective_cv = min(cv, len(groups_train.unique()))
        
        result = linear_reg(model_type,
                            X_train, X_test,
                            y_train, y_test,
                            effective_cv, groups_train,
                            label)
        
        fold_results[dataset_name] = result

    # Aggregate fold results
    final                = fold_results[list(fold_results.keys())[-1]].copy()
    final['test_r2']     = np.mean([r['test_r2']   for r in fold_results.values()])
    final['test_rmse']   = np.mean([r['test_rmse'] for r in fold_results.values()])
    final['fold_sites']  = {k: v['test_r2'] for k, v in fold_results.items()}
    return final

In [ ]:
def linear_reg_ver2(model_type, X_train, X_test,
                    y_train, y_test, cv, groups, label,display=True):
    return linear_reg(model_type,
                      X_train, X_test,
                      y_train, y_test,
                      cv, groups,
                      label, display=True)

In [2]:
def linear_reg(model_type,
               X_train, X_test, y_train, y_test,
               cv, groups, label,display=True):
    # Log-transform
    y_train_log = np.log1p(y_train)
    y_test_log  = np.log1p(y_test)

    # Scaling is necessary for linear regression regardless of whether the target is log-transformed or not.
    scaler  = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    if model_type == 'ridge':
        model    = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
        model_cv = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
        name     = "RIDGE REGRESSION: "
    elif model_type == 'lasso':
        model    = LassoCV(alphas=[0.001, 0.01, 0.1, 1.0], cv=5)
        model_cv = LassoCV(alphas=[0.001, 0.01, 0.1, 1.0], cv=5)
        name     = "LASSO REGRESSION: "
    elif model_type == 'elasticnet':
        model    = ElasticNetCV(alphas=[0.001, 0.01, 0.1, 1.0], l1_ratio=[0.2, 0.5, 0.8], cv=5)
        model_cv = ElasticNetCV(alphas=[0.001, 0.01, 0.1, 1.0], l1_ratio=[0.2, 0.5, 0.8], cv=5)
        name     = "ELASTICNET REGRESSION: "
    else:
        model    = LinearRegression()
        model_cv = LinearRegression()
        name     = "LINEAR REGRESSION: "
    
    model.fit(X_train_scaled, y_train_log)

    return post_processing(model,
                           X_train_scaled, X_test_scaled,
                           y_train, y_test, y_train_log,
                           cv, model_cv, groups,
                           label + ", ALGO: " + name,
                           display)

## RANDOM FOREST FUNCTIONS

In [ ]:
def randomForest_regular(X_var, y_var, cv, label, grid, display=True):
    X_train, X_test, y_train, y_test = split_data_ver2(X_var, y_var)
    return random_forest(X_train, X_test,
                         y_train, y_test,
                         cv, None,
                         label, grid, display)

In [ ]:
def randomForest_groups(X_var, y_var,
                        cv, groups,
                        label, grid, display=True):
    X_train, X_test, y_train, y_test = split_data_by_groups(X_var, y_var, groups)
    
    # Split groups the same way as X and y
    groups_train = groups[y_train.index]
    groups_test  = groups[y_test.index]
    
    return random_forest(X_train, X_test, y_train, y_test,
                         cv, groups_train,
                         label, grid, display)

In [ ]:
def randomForest_ver2(X_train, X_test,
                      y_train, y_test,
                      cv, groups,
                      label, grid, display=True):
    return random_forest(X_train, X_test,
                         y_train, y_test,
                         cv, groups, label, grid)

In [7]:
def random_forest(X_train, X_test, y_train, y_test,
                  cv, groups, label, grid, display=True):
    X_var = pd.concat([X_train, X_test], axis=0)
    y_var = pd.concat([y_train, y_test], axis=0)

    # Log-transform
    y_train_log = np.log1p(y_train)
    y_test_log  = np.log1p(y_test)
    
    # NOTE: Random forest does not need scaling.

    if grid:
        # Hyperparameter tuning
        param_grid = {
            'max_depth'       : [5, 10, 15, None],
            'min_samples_leaf': [2, 4, 8],
            'max_features'    : ['sqrt', 0.3, 0.5],
            'max_samples'     : [0.6, 0.8, 1.0]
        }
        grid_search = GridSearchCV(
            RandomForestRegressor(n_estimators=500,
                                  random_state=42,
                                  n_jobs=-1),
            param_grid,
            cv=cv,
            scoring='r2',
            n_jobs=-1
        )
        grid_search.fit(X_train, y_train_log)
        print(f"Best params: {grid_search.best_params_}")
        
        # Train final model with best params
        rf_model = grid_search.best_estimator_
    
        # Cross-validation to confirm that the R^2 above is not by coincidence.
        rf_cv_model = RandomForestRegressor(**grid_search.best_params_,
                                            n_estimators=500,
                                            random_state=42,
                                            n_jobs=-1)
    else:
        rf_model = RandomForestRegressor(
            n_estimators    = 500,
            max_features    = 'sqrt',
            min_samples_leaf= 2,
            random_state    = 42,
            n_jobs          = -1
        )
        rf_cv_model = RandomForestRegressor(
            n_estimators    = 500,
            max_features    = 'sqrt',
            min_samples_leaf= 2,
            random_state    = 42,
            n_jobs          = -1
        )

        rf_model.fit(X_train, y_train_log)
    
    datum = post_processing(rf_model,
                            X_train, X_test,
                            y_train, y_test, y_train_log,
                            cv, rf_cv_model, groups,
                            "RANDOM FOREST: " + label,
                            display)
    
    importances = pd.Series(rf_model.feature_importances_, index=X_var.columns)
    imp_data = {"importances": importances}
    datum.update(imp_data)

    return datum